In [ ]:
# Cell 1: Install required libraries
!pip install transformers datasets torch scikit-learn accelerate -q

In [ ]:
# Cell 2: Imports
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Cell 3: Load AG News dataset
dataset = load_dataset("ag_news")

print(dataset)
print("\nSample entry:", dataset["train"][0])
print("\nLabel names:", dataset["train"].features["label"].names)

# AG News has 4 classes:
# 0 = World, 1 = Sports, 2 = Business, 3 = Sci/Tech
label_names = dataset["train"].features["label"].names
num_labels = len(label_names)
print(f"\nNumber of classes: {num_labels}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

Sample entry: {'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}

Label names: ['World', 'Sports', 'Business', 'Sci/Tech']

Number of classes: 4


In [ ]:
# Cell 4: Tokenize
MODEL_NAME = "bert-base-uncased"
tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,      # 128 is enough for news headlines
        padding=False        # DataCollator handles padding dynamically
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Rename label column to what Trainer expects
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Tokenization complete!")
print("Train size:", len(tokenized_dataset["train"]))
print("Test size:", len(tokenized_dataset["test"]))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

Tokenization complete!
Train size: 120000
Test size: 7600


In [ ]:
# Cell 5: Subsample for Colab speed (optional — remove for full training)
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(5000))
eval_dataset  = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

print(f"Using {len(train_dataset)} train / {len(eval_dataset)} eval samples")

Using 5000 train / 1000 eval samples


In [ ]:
# Cell 6: Load BERT for sequence classification
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label={i: label_names[i] for i in range(num_labels)},
    label2id={label_names[i]: i for i in range(num_labels)}
)

print("Model loaded!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded!
Parameters: 109,485,316


In [ ]:
# Cell 7: Metrics function
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="weighted")

    return {
        "accuracy": round(acc, 4),
        "f1_weighted": round(f1, 4)
    }

In [ ]:
# Cell 8: Training configuration
from transformers import TrainingArguments
import transformers

print(f"Transformers version: {transformers.__version__}")

training_args = TrainingArguments(
    output_dir="./bert-ag-news",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    warmup_steps=100,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy="epoch",        # renamed from evaluation_strategy
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    logging_dir="./logs",
    logging_steps=50,
    fp16=True,
    report_to="none"
)

print("TrainingArguments created successfully!")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Transformers version: 5.0.0
TrainingArguments created successfully!


In [ ]:
# Cell 9: Initialize Trainer and train
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,      # renamed from 'tokenizer'
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1 Weighted
1,0.366282,0.367490,0.886000,0.885900
2,0.212584,0.297255,0.903000,0.903100
3,0.099447,0.309217,0.915000,0.915300


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=939, training_loss=0.3272578277019139, metrics={'train_runtime': 11102.5399, 'train_samples_per_second': 1.351, 'train_steps_per_second': 0.085, 'total_flos': 707394997156224.0, 'train_loss': 0.3272578277019139, 'epoch': 3.0})

In [ ]:
# Cell 10: Final evaluation
results = trainer.evaluate()

print("\n" + "="*40)
print("FINAL EVALUATION RESULTS")
print("="*40)
print(f"Accuracy  : {results['eval_accuracy']:.4f}")
print(f"F1 Score  : {results['eval_f1_weighted']:.4f}")
print(f"Eval Loss : {results['eval_loss']:.4f}")

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



FINAL EVALUATION RESULTS
Accuracy  : 0.9150
F1 Score  : 0.9153
Eval Loss : 0.3092


In [ ]:
# Cell 11: Inference on new headlines
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

test_headlines = [
    "Stock markets surge as inflation fears ease",
    "Champions League final: Real Madrid defeats Liverpool",
    "NASA launches new Mars exploration rover",
    "UN Security Council meets over Ukraine conflict"
]

print("\nPredictions:")
print("-" * 50)
for headline in test_headlines:
    result = classifier(headline)[0]
    print(f"Text  : {headline}")
    print(f"Label : {result['label']}  |  Score: {result['score']:.4f}")
    print()


Predictions:
--------------------------------------------------
Text  : Stock markets surge as inflation fears ease
Label : Business  |  Score: 0.9874

Text  : Champions League final: Real Madrid defeats Liverpool
Label : Sports  |  Score: 0.9829

Text  : NASA launches new Mars exploration rover
Label : Sci/Tech  |  Score: 0.9909

Text  : UN Security Council meets over Ukraine conflict
Label : World  |  Score: 0.9950



In [ ]:
# Cell 12: Save model to Google Drive (optional)
# from google.colab import drive
# drive.mount('/content/drive')
# model.save_pretrained("/content/drive/MyDrive/bert-ag-news")
# tokenizer.save_pretrained("/content/drive/MyDrive/bert-ag-news")

# Or save locally
model.save_pretrained("./bert-ag-news-final")
tokenizer.save_pretrained("./bert-ag-news-final")
print("Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved!
